# ReFuelEU optimisation — results

The paper's figures, drawn from what `01_optimisation_runs.ipynb` leaves in
`results/` — the JSON outputs, and for section 3 the optimisation histories. No model
runs here, so this notebook is seconds, not hours.

Sections 1 and 6 need only the `main` case; 2 and 3 need `main`'s ten budgets; 4 needs
all four biomass cases; 5 needs `main` and `pess`. Each section says what is missing rather
than failing, so it is usable while the sweeps are still running.

**Regenerated for the revision.** These figures no longer read `results/`. They read
`sweep/results_e/`, which is block E of the overnight sweep, and two things moved under
the published surface:

- aviation's biomass allocation is a round **10 %**, not the reverse-engineered 9.90 %.
  The old value was chosen so that production efficiency covered 2019 aviation energy
  use, which makes a convention look derived;
- the surplus β is anchored on `initial_airfare_per_rpk` — the same 2019 price that
  anchors the inverse supply function and the traffic response — where it used to be
  anchored on the scenario's own 2025 airfare. That moves the objective non-uniformly,
  because the surplus share of it runs from 15 % to 98 % across the budget ladder, so
  optima move rather than merely rescale;
- the ramp-up limit (Eq. 12) reads its volume branch as an *increment*,
  `max{E_{t-1}(1+τ)^Δt, E_{t-1} + ΔE·Δt}`, as the equation writes it. The code used to
  read it as an absolute ceiling `ΔE·Δt`, which silently turned into a pure rate limit on
  any pathway already past one period's allowance. The correction only ever loosens;
- mandates step to their first value in 2025 instead of ramping linearly from 2020. The
  ramp burned biofuel over 2021–2024 inside the ReFuelEU run that *defines* the budget,
  tightening it by 0.105 %: the budget moves from 3.8616 to 3.8656 GtCO₂.

Every surplus number in the submitted paper therefore has to be re-read off these runs:
Table 3, the 614 Bn€ against BAU and its 537 Bn€ / 27 % split, the ReFuelEU annotations
on Figure 9, and §4.4's 39 % / 242 Bn€ / 95 Bn€.

The one-parameter sensitivities of blocks A–D live in `sweep/` instead, with their own
figures; they are held at a single carbon budget and are not part of this ladder.

**One conversion changed.** Legacy MFSPs were €/L and the figures divided them by 35 to
reach €/MJ. On `main` they are stored in €/MJ already, so that division is gone — see
`00_migration_validation.ipynb` §1 on why 35.3, not 35 or 35.2, is the number the paper
actually used.

## 0. Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.legend_handler import HandlerTuple
from scipy.interpolate import griddata

import optimisation_runs as R

# Block E of the overnight sweep, which supersedes the published ladder -- see the
# heading above for what moved. The original runs are still in ../results and can be
# read by pointing this back, but they are not comparable with anything under sweep/.
R.RESULTS_DIR = Path("sweep/results_e")

# Cumulative 2020-2050 CO2 of ReFuelEU (linear) at eps_P = -0.9, as a share of the world
# aviation budget: 3.8656 GtCO2 on the EU perimeter. Blocks A-D were all held exactly
# here, so on the budget axis of Figure 9 it marks where the regulation's own ambition
# falls, between the 3.2 and 3.0 rungs of the ladder.
REFUELEU_BUDGET_WORLD_SHARE = 3.122636344

warnings.filterwarnings("ignore")

BUDGETS = [2.0, 2.2, 2.4, 2.6, 2.8, 3.0, 3.2, 3.4, 3.6, 3.8]
YEARS = np.array(R.YEARS)

# Paper colours.
GREEN, BLUE, RED, GREY = "#7e9b59", "#092054", "#cb3629", "lightgrey"


def available(case, tag):
    """Path of a saved run, or None if that run has not been done yet."""
    stem = f"{tag}_{case}" if tag in ("fossil", "refueleu") else f"opt_{case}_{tag}"
    path = R.RESULTS_DIR / f"{stem}.json"
    return path if path.exists() else None


def collect(case, tags):
    """Load every run of `case` that exists, keyed by tag."""
    out = {}
    for tag in tags:
        path = available(case, tag)
        if path:
            out[tag] = R.load(path)
    missing = [t for t in tags if t not in out]
    if missing:
        print(f"{case}: not on disk yet -> {', '.join(missing)}")
    return out


ALL_TAGS = ["mincarb"] + [R.budget_tag(b) for b in BUDGETS]

# --- House style ---------------------------------------------------------------------
# The look section 2b settles on, set once here rather than repeated per figure. Titles
# go bold and left everywhere, including the figures section 7 draws from `sweep/`.
# One type scale across the notebook's figures, so two of them side by side in the paper
# do not arrive at different sizes: a full-size panel's title and axis labels at
# PANEL_LABEL_SIZE, the smaller panels of a mixed row at SMALL_LABEL_SIZE, and everything
# secondary -- ticks, legends -- at SECONDARY_SIZE.
PANEL_LABEL_SIZE = 13
SMALL_LABEL_SIZE = 11
SECONDARY_SIZE = 9

plt.rcParams.update(
    {
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "figure.titleweight": "bold",
        "axes.titlesize": PANEL_LABEL_SIZE,
        "axes.labelsize": SMALL_LABEL_SIZE,
        "xtick.labelsize": SECONDARY_SIZE,
        "ytick.labelsize": SECONDARY_SIZE,
        "legend.fontsize": SECONDARY_SIZE,
    }
)


def tidy(*axes):
    """Open the frame at the top and right, mute what is left, rule the y axis only.

    For line plots. The image figures -- the constraint heatmap, the trade-off surface,
    the abatement surface -- keep their full frame, since half a frame drawn round an
    image reads as a broken one rather than as a light touch.

    Takes loose axes or whatever `plt.subplots` returned, in any mix.
    """
    for group in axes:
        for ax in np.ravel(group) if isinstance(group, np.ndarray) else [group]:
            for side in ("top", "right"):
                ax.spines[side].set_visible(False)
            for side in ("left", "bottom"):
                ax.spines[side].set_color("0.4")
            ax.tick_params(color="0.4")
            # Clear whatever the figure asked for first: a grid already switched on for
            # both axes keeps its verticals when only the y axis is restyled.
            ax.grid(False)
            ax.grid(True, axis="y", color="0.88", lw=0.8)
            ax.set_axisbelow(True)

## 1. One optimised run against the fossil BAU

The headline numbers of §4.1, then Figure 6: the mandate the optimiser chose, and how
close each pathway sits to the two limits that bound it.

**Read at the ReFuelEU-equivalent budget**, not at a rung of the ladder, so that the
optimum discussed in the text and the regulation it is compared against consume the same
carbon: 3.8656495 GtCO₂ cumulative 2020–2050, or 3.122636344 % of the world budget.

That value falls between the 3.2 and 3.0 rungs, so it has its own optimisation —
`sweep/run_refueleu_budget.py`, saved as `opt_main_refueleu`. Same case as the ladder:
`main` at 10 % biomass, ε_P = −0.9, r = 4.5 %, default ramp-up caps, warm-started from the
3.2 optimum.

Blocks A–D contain an optimisation at the same budget, `base`, reached from a different
start — the previous sweep's optimum. On a non-convex problem the start point can decide
which local optimum SLSQP finds, so the two are worth comparing rather than assuming:
they agree to 6.5e-05 percentage points on the design and 1.1e-05 on the objective. The script prints
that check every time it runs.

At 2.8 % the same comparison gives a far more dramatic scenario — a 93.3 % blend, airfare
up 31.4 %, traffic at 78.2 % of BAU — because 2.8 % is a materially tighter budget than
the regulation implies. Quoting those numbers beside ReFuelEU would overstate what the
regulation itself asks for.

In [ ]:
# The ReFuelEU-equivalent budget: 3.8656 GtCO2 cumulative 2020-2050, or 3.122636 % of
# the world aviation budget. Its own run -- see the heading above for why, and for the
# cross-check against the cold-started optimisation at the same budget.
opt, _ = R.load(R.RESULTS_DIR / "opt_main_refueleu.json")
bau, _ = R.load(R.RESULTS_DIR / "fossil_main.json")

pd.DataFrame(
    [
        [
            "Airline cost per RPK",
            opt["total_cost_per_rpk"].loc[2050] / bau["total_cost_per_rpk"].loc[2050],
        ],
        ["Airfare per RPK", opt["airfare_per_rpk"].loc[2050] / bau["airfare_per_rpk"].loc[2050]],
        ["Traffic vs no elasticity", opt["rpk"].loc[2050] / opt["rpk_no_elasticity"].loc[2050]],
        ["Traffic vs BAU", opt["rpk"].loc[2050] / bau["rpk"].loc[2050]],
        [
            "Annual CO2 vs BAU",
            opt["co2_emissions_including_energy"].loc[2050]
            / bau["co2_emissions_including_energy"].loc[2050],
        ],
        [
            "Cumulative CO2 vs BAU",
            opt["cumulative_co2_emissions"].loc[2050] / bau["cumulative_co2_emissions"].loc[2050],
        ],
        [
            "RPK CAGR 2025-2050 (%)",
            ((opt["rpk"].loc[2050] / opt["rpk"].loc[2025]) ** (1 / 25) - 1) * 100,
        ],
        [
            "RPK CAGR 2025-2050, BAU (%)",
            ((bau["rpk"].loc[2050] / bau["rpk"].loc[2025]) ** (1 / 25) - 1) * 100,
        ],
    ],
    columns=["2050 indicator", "value"],
).round(4).to_string(index=False)

In [ ]:
# Figure 6 -- mandate, and each pathway against its ramp-up and resource envelopes.
fig, (ax_mandate, ax_bio, ax_ele) = plt.subplots(1, 3, figsize=(15, 5))

# --- left: the chosen mandate, against the regulation as written and as legislated ---
ax_mandate.plot(YEARS, opt["generic_biofuel_share_dropin_fuel"], color=GREEN, lw=2)
ax_mandate.plot(YEARS, opt["generic_electrofuel_share_dropin_fuel"], color=BLUE, lw=2)
ax_mandate.plot(YEARS, opt["fossil_kerosene_share_dropin_fuel"], color=RED, lw=2)

refueleu_years = [2020, 2025] + R.OPTIM_YEARS
refueleu_bio = [0, 2] + R.REFUELEU_MANDATE["biofuel"]
refueleu_ele = [0, 0] + R.REFUELEU_MANDATE["electrofuel"]
ax_mandate.plot(refueleu_years, refueleu_bio, "--", color=GREEN, lw=1.5)
ax_mandate.plot(refueleu_years, refueleu_ele, "--", color=BLUE, lw=1.5)
ax_mandate.plot(
    refueleu_years,
    [100 - b - e for b, e in zip(refueleu_bio, refueleu_ele)],
    "--",
    color=RED,
    lw=1.5,
)

# ReFuelEU as legislated: flat steps between reference years, not a linear ramp.
# Derived from the legislated totals (6 % SAF of which 1.2 % synthetic in 2030-34, and
# so on) rather than the hand-entered series the published figure carried.
step_bio = pd.Series(0.0, index=R.YEARS)
step_ele = pd.Series(0.0, index=R.YEARS)
for start, end, bio, ele in [
    (2025, 2030, 2, 0),
    (2030, 2035, 6, 1.2),
    (2035, 2040, 20, 5),
    (2040, 2045, 34, 10),
    (2045, 2050, 42, 15),
    (2050, 2051, 70, 35),
]:
    step_bio.loc[start : end - 1] = bio - ele
    step_ele.loc[start : end - 1] = ele
ax_mandate.plot(YEARS, step_bio, ":", color=GREEN, lw=1.5)
ax_mandate.plot(YEARS, step_ele, ":", color=BLUE, lw=1.5)
ax_mandate.plot(YEARS, 100 - step_bio - step_ele, ":", color=RED, lw=1.5)

ax_mandate.legend(
    handles=[
        plt.Line2D([], [], color=GREEN, lw=1.5, label="Biofuel"),
        plt.Line2D([], [], color=BLUE, lw=1.5, label="Electrofuel"),
        plt.Line2D([], [], color=RED, lw=1.5, label="Fossil"),
        plt.Line2D([], [], color="k", lw=1.5, ls="-", label="Optimisation"),
        plt.Line2D([], [], color="k", lw=1.5, ls="--", label="ReFuelEU (linear)"),
        plt.Line2D([], [], color="k", lw=1.5, ls=":", label="ReFuelEU (step)"),
    ],
    loc="center left",
)
ax_mandate.set_ylabel("Drop-in fuel shares (%)", fontsize=PANEL_LABEL_SIZE)
ax_mandate.set_xlim(2021, 2050)
ax_mandate.set_ylim(0, 105)
ax_mandate.grid(alpha=0.3)
ax_mandate.set_title("Blending mandate")


def envelopes(consumption, rate=0.2, volume=0.2 * R.EU_ASK_SHARE):
    """The two ramp-up limits of Eq. 12, as the sawtooth the paper plots.

    Each reference year gets two points at the same abscissa: the cap implied by the
    previous year, then what the scenario actually consumed. The feasible set is the
    *larger* of the two caps, hence the max when shading.
    """
    years, caps_rate, caps_volume = (
        [2025],
        [consumption.loc[2025] / 1e12],
        [consumption.loc[2025] / 1e12],
    )
    for year in R.OPTIM_YEARS:
        previous = consumption.loc[year - 5]
        years += [year, year]
        caps_rate += [previous * (1 + rate) ** 5 / 1e12, consumption.loc[year] / 1e12]
        caps_volume += [(previous + volume * 5 * 1e12) / 1e12, consumption.loc[year] / 1e12]
    return years, caps_rate, caps_volume


for ax, pathway, origin, colour, title in [
    (ax_bio, "generic_biofuel", "biomass", GREEN, "Biofuel"),
    (ax_ele, "generic_electrofuel", "electricity", BLUE, "Electrofuel"),
]:
    consumption = opt[f"{pathway}_energy_consumption"]
    # Resource envelope in fuel terms: what the aviation allocation can actually make.
    resource_cap = (
        opt[f"{origin}_availability_aviation_allocated"] / R.RESOURCE_PER_FUEL[origin] / 1e12
    )

    years, caps_rate, caps_volume = envelopes(consumption)
    caps = np.maximum(caps_rate, caps_volume)

    ax.plot(YEARS, consumption / 1e12, color=colour, lw=3, label="Actual consumption")
    ax.plot(years, caps_rate, ":", color="#CCCCCC", lw=3, label="Ramp-up (rate)")
    ax.plot(years, caps_volume, "--", color="#CCCCCC", lw=2, label="Ramp-up (volume)")
    ax.plot(YEARS, resource_cap, "-", color="#CCCCCC", lw=2, label="Resource constraint")

    # Infeasible above either limit; feasible below both.
    ax.fill_between(YEARS, resource_cap, 3, facecolor=RED, alpha=0.1, lw=0)
    ax.fill_between(years, caps, 3, facecolor=RED, alpha=0.1, lw=0)
    ax.fill_between(
        years,
        0,
        np.minimum(caps, np.interp(years, YEARS, resource_cap)),
        facecolor=GREEN,
        alpha=0.1,
        lw=0,
    )

    ax.set_ylabel("Consumption [EJ]", fontsize=PANEL_LABEL_SIZE)
    ax.set_xlim(2025, 2050)
    ax.set_ylim(-0.05, 1.5)
    ax.grid()
    ax.legend(loc="upper left")
    ax.set_title(title)

tidy(ax_mandate, ax_bio, ax_ele)
fig.tight_layout()
fig.savefig("ressource_constraints.pdf")

## 2. Carbon-budget sensitivity — the reference case

Figures 7 and 8: what the optimiser does with the mandate as the budget tightens, and
what that costs in traffic and airfare. Three references are drawn against the ladder and
named once, in a legend inside the traffic panel: ReFuelEU in red, the fossil BAU dotted
in grey, and in green section 1's reference case — `opt_main_refueleu`, the optimum run
*at* the ReFuelEU-equivalent budget. Green against red is the comparison §4.1 is about:
same carbon, different mandate.

The colourbar is a continuous budget axis, and colour is one mapping from the share of
the world carbon budget a run consumes — the same `budget_norm` colours the curves and
fills the bar, so a curve's shade can be read straight off it. Every scenario is a tick, each
crossed by a hairline so it marks an exact position on the gradient rather than a label
floating beside it: Min CO₂ at 2.10, then the rungs 2.2 to 3.8. The 2.0 target is infeasible — it lands on
Min CO₂ at 2.10 — so those two runs share one tick rather than implying a scenario
between them that does not exist.

None of the three references is a rung, so each is a line where it stands rather than a
tick, with the names in a column of their own clear of the numbers. The optimum and
ReFuelEU share the same 3.1226 to seven decimals — necessarily, the optimum having been
run at that budget — so their two lines coincide and only the labels are separated. BAU
is at 4.10: the gradient stops with the ladder at 3.9 and the axis carries on past it, so
BAU's mark sits in blank space rather than under a shade that would imply runs up there.

The black dashed reference is one counterfactual seen twice: fares frozen at their 2019
level, so traffic never responds to price. That constant is the elasticity's own anchor
(`RPKElasticity.REFERENCE_AIRFARE_PER_RPK`), and the model is
`rpk = rpk_no_elasticity * (airfare / initial_airfare) ** price_elasticity`, so a scenario
holding the fare there lands exactly on the no-elasticity traffic — confirmed on these
runs, where the implied exponent is −0.900000 in every year of every case. The traffic
panel shows the quantity side of it and the airfare panel the price side, which is why
they share one entry.

One caveat for reading it: the multiplier is clamped to 1 up to and including the latest
`covid_end_year`, so over 2020–2024 the two agree *whatever* the fare — in 2020 the fare
is 32 % above the anchor and traffic is unmoved. Only from 2025 does the agreement carry
the meaning above.

In [ ]:
runs = collect("main", ALL_TAGS)
refueleu = R.load(R.RESULTS_DIR / "refueleu_main.json") if available("main", "refueleu") else None
bau = R.load(R.RESULTS_DIR / "fossil_main.json") if available("main", "fossil") else None

# Section 1's reference case: the optimum run *at* the ReFuelEU-equivalent budget. It is
# its own optimisation rather than a rung of the ladder, so `available` -- which maps the
# "refueleu" tag to the un-optimised `refueleu_main` -- cannot name it.
_optimum_path = R.RESULTS_DIR / "opt_main_refueleu.json"
optimum = R.load(_optimum_path) if _optimum_path.exists() else None

ordered = [t for t in ALL_TAGS if t in runs]
labels = {"mincarb": "Min $CO_2$", **{R.budget_tag(b): f"{b}" for b in BUDGETS}}
cmap = plt.get_cmap("Blues")

DARK_GREY = "#555555"


def consumed(run):
    """Share of the *world* carbon budget a run consumes -- the colourbar's own scale."""
    return run[1]["carbon_budget_consumed_share"] / R.EU_ASK_SHARE


budget_of = {tag: consumed(runs[tag]) for tag in ordered}
bau_budget = consumed(bau) if bau is not None else None
refueleu_budget = consumed(refueleu) if refueleu is not None else REFUELEU_BUDGET_WORLD_SHARE
optimum_budget = consumed(optimum) if optimum is not None else None

# --- One budget-to-colour mapping, shared by the curves and the colourbar -------------
# A continuous ramp rather than one band per run, with every scenario a tick on it. Blues
# is truncated to 0.3-1.0 so the lightest run still reads against white. The ramp spans
# the ladder; the bar's axis is stretched past it further down, so BAU's mark has blank
# space to sit in rather than a shade implying runs that do not exist.
BAR_MIN = min(budget_of.values()) - 0.05
BAR_MAX = max(budget_of.values()) + 0.1
budget_norm = mcolors.Normalize(vmin=BAR_MIN, vmax=BAR_MAX)
budget_cmap = mcolors.LinearSegmentedColormap.from_list(
    "budget_blues", cmap(np.linspace(0.3, 1.0, 256))
)
colours = {tag: budget_cmap(budget_norm(b)) for tag, b in budget_of.items()}

fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex="col")

panels = [
    (
        axs[0, 0],
        "generic_biofuel_share_dropin_fuel",
        "Biofuel Share (%)",
        "Biofuel Share",
        (-1, 59),
    ),
    (
        axs[1, 0],
        "generic_electrofuel_share_dropin_fuel",
        "Electrofuel Share (%)",
        "Electrofuel Share",
        (-1, 59),
    ),
    (axs[0, 1], "rpk", "RPK", "Traffic", None),
    (axs[1, 1], "airfare_per_rpk", "Airfare per RPK (€)", "Airfare", (0.06, 0.15)),
]

for ax, key, ylabel, title, ylim in panels:
    for tag in ordered:
        ax.plot(YEARS, runs[tag][0][key], color=colours[tag], lw=1.8)
    if bau is not None:
        ax.plot(YEARS, bau[0][key], color=DARK_GREY, ls=":", lw=2.2)
    if refueleu is not None:
        ax.plot(YEARS, refueleu[0][key], color=RED, ls="--", lw=2)
    if optimum is not None:
        ax.plot(YEARS, optimum[0][key], color=GREEN, ls="--", lw=2)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xlim(2019)
    if ylim:
        ax.set_ylim(ylim)
    ax.grid(True)

# One counterfactual seen twice: fares frozen at their 2019 level, so traffic never
# responds to price. `rpk_no_elasticity` is identical in every run, and every scenario's
# 2019 airfare is exactly the anchor -- the traffic panel shows the quantity side of it
# and the airfare panel the price side, which is why they share a legend entry.
axs[0, 1].plot(YEARS, runs[ordered[-1]][0]["rpk_no_elasticity"], color="black", ls="--")
axs[0, 1].set_ylim(0)
axs[1, 1].axhline(0.09236379319842411, color="black", ls="--")

axs[1, 0].set_xlabel("Year")
axs[1, 1].set_xlabel("Year")

# --- One legend for the whole figure, in the traffic panel ---------------------------
axs[0, 1].legend(
    handles=[
        mlines.Line2D([], [], color=DARK_GREY, ls=":", lw=2.2, label="BAU"),
        mlines.Line2D([], [], color=RED, ls="--", lw=2, label="ReFuelEU (linear)"),
        mlines.Line2D([], [], color=GREEN, ls="--", lw=2, label="Optimum at ReFuelEU budget"),
        mlines.Line2D([], [], color="black", ls="--", label="No elasticity / 2019 airfare"),
    ],
    loc="lower right",
    frameon=True,
    fontsize=9,
)

# --- Colourbar: a continuous budget axis, every scenario a tick on it ----------------
# Positions are the realised budgets, not the targets. None of the three references is a
# scenario of the ladder, so each is a line where it stands rather than a tick.
# The 2.0 target is infeasible: it lands on Min CO2 at 2.10, so on a budget axis the two
# runs are the same point and share one tick rather than implying a scenario between them
# that does not exist.
mincarb_budget = budget_of.get("mincarb")
sorted_tags = sorted(
    (
        t
        for t in ordered
        if t == "mincarb" or mincarb_budget is None or abs(budget_of[t] - mincarb_budget) >= 0.05
    ),
    key=lambda t: budget_of[t],
)
scalar = plt.cm.ScalarMappable(cmap=budget_cmap, norm=budget_norm)

fig.subplots_adjust(right=0.82)
# A continuous norm is linear in budget by construction, so no `spacing` argument is
# needed here -- that only governs how a BoundaryNorm's bands are sized.
cbar = fig.colorbar(scalar, cax=fig.add_axes([0.85, 0.15, 0.02, 0.7]))
# The gradient covers the ladder; the axis reaches beyond it so BAU's mark at 4.10 has
# somewhere to sit.
if bau is not None:
    cbar.ax.set_ylim(BAR_MIN, bau_budget + 0.1)
# Pushed out past the reference-name column, which matplotlib does not measure.
cbar.set_label("Share of world carbon budget (%)", labelpad=15, fontsize=12)

tick_values = [budget_of[t] for t in sorted_tags]
cbar.set_ticks(tick_values)
cbar.set_ticklabels([labels[t] for t in sorted_tags])
cbar.ax.tick_params(length=0)

# A hairline across the bar at each scenario, so a tick reads as an exact position on the
# gradient rather than a label floating beside it. Each takes whichever of white or near
# black carries contrast against the shade it crosses, the ramp running light to dark.
for value in tick_values:
    r, g, b, _ = budget_cmap(budget_norm(value))
    luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
    cbar.ax.axhline(value, color="white" if luminance < 0.5 else "#222222", lw=0.8)


# Neither reference is a rung -- ReFuelEU at 3.12 falls between the 3.0 and 3.2 levels,
# BAU at 4.10 sits beyond the top of the ladder -- so each is drawn at exactly where it
# stands, in the style its curve carries in the panels.
#
# The optimum was run *at* the ReFuelEU-equivalent budget, so its mark and ReFuelEU's land
# on the same 3.1226 to seven decimals. Both lines are drawn there -- the optimum's dashes
# sparse enough to let the red show through -- and only the labels are nudged apart, since
# moving a line would put it where no run stands.
# The names sit in a column of their own, clear of the numeric ticks: a label at 3.12
# otherwise lands on the "3.2" tick, and the two cannot simply be stacked between the 3.0
# and 3.2 ticks -- that gap is about 29 px and two 10 px labels do not fit in it with any
# clearance left.
#
# The column is in axes fractions of a bar only ~26 px wide, so 0.1 here is 2.6 px: the
# tick labels reach x = 1.9 themselves, and the column has to start beyond them.
LABEL_COLUMN = 1.2


def mark(value, text, colour, ls, lw, label_offset=0.0):
    """Put one reference on the colourbar, at its own budget."""
    cbar.ax.axhline(value, color=colour, lw=lw, ls=ls)
    cbar.ax.text(
        LABEL_COLUMN,
        value + label_offset,
        text,
        color=colour,
        fontsize=9,
        va="center",
        ha="left",
        transform=cbar.ax.get_yaxis_transform(),
    )


if refueleu is not None:
    mark(refueleu_budget, "ReFuelEU", RED, "--", 1.8, label_offset=0.015)
if optimum is not None:
    mark(optimum_budget, "Ref. optim.", GREEN, (0, (2, 4)), 2.0, label_offset=-0.035)
if bau is not None:
    mark(bau_budget, "BAU", DARK_GREY, ":", 2.2)

tidy(axs)
plt.tight_layout(rect=[0, 0, 0.84, 1])
plt.savefig("overall.pdf", bbox_inches="tight")

### 2b. The same runs, arranged differently

A second version of the figure above, not a replacement — run that cell first, since this
one reuses its data, its budget-to-colour mapping and its ticks rather than rebuilding
them, which is what keeps the two in step.

The two mandates take the top row at full width: they are what the optimiser actually
chooses. The row below is what that choice costs — CO₂, airfare, traffic — on smaller
axes. CO₂ is `co2_emissions_including_energy`, the series whose 2020–2050 sum is the
quantity the carbon budget is written in: 3865.65 MtCO₂ for the ReFuelEU-equivalent run,
against its 3.8656 GtCO₂ target.

Presentation: top and right spines dropped, a horizontal rule only behind the lines,
larger y-axis labels, and traffic in billions so the axis carries plain numbers rather
than an `1e12` offset drawn over the title. CO₂ needs no scaling for the same reason —
the series is already in Mt. The legend moves to the biofuel panel, the bottom row now being too narrow to
hold four entries without covering curves.

Both reference curves carry markers at the five ReFuelEU reference years, which are the
regulation's milestones and the optimiser's design variables alike — circles for the
mandate as written, squares for the optimum, since the two coincide in places.

The third row folds in what was section 3's `constraints_active_set` figure: for each
budget, how much of every limit the run actually uses, normalised so 1.0 is the limit itself, a dash where a constraint is
active and a red one where it is past its limit. Its rows are the same budget ladder the
curves above are coloured by, which is what makes the merge worth doing — one organising
variable for the whole figure, read as *what the optimiser chose, what it cost, and what
stopped it*.

Two things differ from the published version. The mandate heatmap that sat beside it is
dropped: the top row already carries those mandates as curves, at annual resolution
rather than at five milestone years. And the saturation scale is grey, not blue — in this
figure blue already means the carbon budget, and a second blue scale meaning something
else would read as the same encoding. Each colourbar spans only the rows it describes.

This row reads the optimisation histories (the HDFs beside the results), not the saved
outputs, so it needs those files; a budget whose history is missing is left out with a
note rather than failing.

In [ ]:
# --- An alternative arrangement of the same figure -----------------------------------
# The two mandates take the top row: they are what the optimiser actually chooses, and
# the three panels below are what that choice costs. Everything the figure above settles
# -- the budget-to-colour mapping, the three references, the ticks -- is reused rather
# than rebuilt, so the two cannot drift apart.
#
# CO2 is `co2_emissions_including_energy`, the series whose 2020-2050 sum is the number
# the carbon budget is written in: 3865.65 MtCO2 for the ReFuelEU-equivalent run, against
# its 3.8656 GtCO2 target.
MANDATE_LABEL_SIZE = PANEL_LABEL_SIZE
COST_LABEL_SIZE = SMALL_LABEL_SIZE

# The five ReFuelEU reference years carry the mandate's milestones and the optimiser's
# design variables alike, so both reference curves are marked there. `markevery` counts
# in samples, not years, hence the lookup into YEARS.
MARK_AT = [int(np.flatnonzero(YEARS == year)[0]) for year in R.OPTIM_YEARS]

# The two mandate panels: wide enough for a tick every five years, which is also the
# spacing of the milestones the markers sit on. The three below get one every ten.
MANDATE_KEYS = {
    "generic_biofuel_share_dropin_fuel",
    "generic_electrofuel_share_dropin_fuel",
}

# Cell 11 rebinds `colours` for its own figure, so this one keeps its own copy: re-running
# this cell after that one would otherwise draw the curves in the wrong colours.
panel_colours = {tag: budget_cmap(budget_norm(b)) for tag, b in budget_of.items()}

fig2 = plt.figure(figsize=(14, 13.5))
gs = fig2.add_gridspec(3, 6, height_ratios=[1.25, 1.0, 1.35])

ax_bio = fig2.add_subplot(gs[0, 0:3])
ax_ele = fig2.add_subplot(gs[0, 3:6])
ax_co2 = fig2.add_subplot(gs[1, 0:2])
ax_fare = fig2.add_subplot(gs[1, 2:4])
ax_rpk = fig2.add_subplot(gs[1, 4:6])
ax_heat = fig2.add_subplot(gs[2, 0:6])

layout = [
    (
        ax_bio,
        "generic_biofuel_share_dropin_fuel",
        "Biofuel share (%)",
        "Biofuel mandate",
        (-1, 59),
        MANDATE_LABEL_SIZE,
        1.0,
    ),
    (
        ax_ele,
        "generic_electrofuel_share_dropin_fuel",
        "Electrofuel share (%)",
        "Electrofuel mandate",
        (-1, 59),
        MANDATE_LABEL_SIZE,
        1.0,
    ),
    (
        ax_co2,
        "co2_emissions_including_energy",
        "MtCO$_2$",
        "CO$_2$ emissions",
        (0, None),
        COST_LABEL_SIZE,
        1.0,
    ),
    (ax_fare, "airfare_per_rpk", "€ / RPK", "Airfare", (0.06, 0.15), COST_LABEL_SIZE, 1.0),
    # Billions, so the axis carries plain numbers: matplotlib's "1e12" offset is drawn
    # above the axes, where it lands on a left-aligned title. CO2 above needs no scaling
    # for the same reason -- the series is already in Mt.
    (ax_rpk, "rpk", "Bn RPK", "Traffic", (0, None), COST_LABEL_SIZE, 1e9),
]

for ax, key, ylabel, title, ylim, size, scale in layout:
    for tag in ordered:
        ax.plot(YEARS, runs[tag][0][key] / scale, color=panel_colours[tag], lw=1.6)
    if bau is not None:
        ax.plot(YEARS, bau[0][key] / scale, color=DARK_GREY, ls=":", lw=2.2)
    if refueleu is not None:
        ax.plot(
            YEARS,
            refueleu[0][key] / scale,
            color=RED,
            ls="--",
            lw=2,
            marker="o",
            markevery=MARK_AT,
            ms=4.5,
        )
    if optimum is not None:
        # A different shape, not just a different colour: the two coincide in places.
        ax.plot(
            YEARS,
            optimum[0][key] / scale,
            color=GREEN,
            ls="--",
            lw=2,
            marker="s",
            markevery=MARK_AT,
            ms=4.5,
        )

    ax.set_ylabel(ylabel, fontsize=size)
    ax.set_title(title, fontsize=size, loc="left", pad=8, fontweight="bold")
    is_mandate = key in MANDATE_KEYS
    ax.set_xlim(2020 if is_mandate else 2019, 2050)
    ax.set_xticks(range(2020, 2051, 5 if is_mandate else 10))
    ax.set_ylim(*ylim)

    tidy(ax)

# The counterfactual, on the two panels where it means something.
ax_rpk.plot(YEARS, runs[ordered[-1]][0]["rpk_no_elasticity"] / 1e9, color="black", ls="--")
ax_fare.axhline(0.09236379319842411, color="black", ls="--")

# One legend, in the biofuel panel: the top row is where there is room for four entries,
# and the panels below are now too narrow to hold it without covering curves.
ax_bio.legend(
    handles=[
        mlines.Line2D([], [], color=DARK_GREY, ls=":", lw=2.2, label="BAU"),
        mlines.Line2D(
            [], [], color=RED, ls="--", lw=2, marker="o", ms=4.5, label="ReFuelEU (linear)"
        ),
        mlines.Line2D([], [], color=GREEN, ls="--", lw=2, marker="s", ms=4.5, label="Ref. optim."),
        mlines.Line2D([], [], color="black", ls="--", label="No elasticity / 2019 airfare"),
    ],
    loc="upper left",
    frameon=False,
    fontsize=SECONDARY_SIZE,
)


# --- Row 3: which constraints bind, and when -----------------------------------------
# Read from the optimisation histories rather than the saved outputs, so this row needs
# the HDFs beside the results; rows whose history is missing are simply left out.
#
# The published figure pairs this with a second heatmap of the mandate that produced it.
# That is dropped here: the top row already carries the same mandates as curves, at annual
# resolution rather than at five milestone years, so it said nothing this figure does not.
HEAT_ROWS = [R.budget_tag(b) for b in sorted(BUDGETS, reverse=True)] + ["mincarb"]
histories = {}
for tag in HEAT_ROWS:
    hdf = R.RESULTS_DIR / f"opt_main_{tag}.hdf"
    read = R.read_constraints(hdf) if hdf.exists() else None
    if read is not None:
        histories[tag] = read
heat_rows = [tag for tag in HEAT_ROWS if tag in histories]
if len(heat_rows) < len(HEAT_ROWS):
    print("no history on disk yet ->", ", ".join(t for t in HEAT_ROWS if t not in histories))

# Every constraint is written as (use - limit) / limit, so 1 + g is the share of the limit
# used and 1.0 is the limit itself.
saturation = np.full((len(heat_rows), len(R.CONSTRAINT_LABELS) * len(R.OPTIM_YEARS) + 1), np.nan)
for row, tag in enumerate(heat_rows):
    run = histories[tag]
    for block, name in enumerate(R.CONSTRAINT_LABELS):
        if name in run["constraints"]:
            saturation[row, block * 5 : block * 5 + 5] = 1 + run["constraints"][name].values
    saturation[row, -1] = 1 + run["carbon"]

# Grey, not blue: in this figure blue already means the carbon budget, on the curves above
# and on their colourbar. A second blue scale meaning something else would read as the same
# encoding.
ACTIVE = 1e-3
GREYS = plt.get_cmap("Greys")
# One definition per mark, used by both the grid and the legend, so the two cannot drift.
# An active cell sits at the limit, which on this ramp is always dark, hence a white mark;
# the outline is what carries that same mark on the legend's white ground.
ACTIVE_MARK = {
    "marker": "_",
    "color": "white",
    "ms": 9,
    "mew": 2,
    "path_effects": [pe.withStroke(linewidth=3.2, foreground="black")],
}
VIOLATED_MARK = {
    "marker": "_",
    "color": RED,
    "ms": 9,
    "mew": 2.4,
    "path_effects": [pe.withStroke(linewidth=3.2, foreground="white")],
}
heat = ax_heat.imshow(saturation, aspect="auto", cmap=GREYS, vmin=0, vmax=1.25)
for row in range(saturation.shape[0]):
    for column in range(saturation.shape[1]):
        value = saturation[row, column]
        if np.isnan(value):
            continue
        if value > 1 + ACTIVE:
            ax_heat.plot(column, row, **VIOLATED_MARK)
        elif value >= 1 - ACTIVE:
            ax_heat.plot(column, row, **ACTIVE_MARK)

years = [str(year) for year in R.OPTIM_YEARS]
blocks = list(R.CONSTRAINT_LABELS.values()) + ["Carbon\nbudget"]
ax_heat.set_xticks(np.arange(saturation.shape[1]))
ax_heat.set_xticklabels(years * len(R.CONSTRAINT_LABELS) + ["all"], rotation=90, fontsize=7)
ax_heat.set_xticks(np.arange(saturation.shape[1] + 1) - 0.5, minor=True)
ax_heat.set_yticks(np.arange(len(heat_rows) + 1) - 0.5, minor=True)
ax_heat.grid(which="minor", color="white", lw=0.8)
ax_heat.tick_params(which="minor", length=0)

for row, tag in enumerate(heat_rows):
    if not histories[tag]["feasible"]:
        ax_heat.add_patch(
            plt.Rectangle(
                (-0.5, row - 0.5),
                saturation.shape[1],
                1,
                fill=False,
                hatch="///",
                edgecolor=RED,
                lw=0,
            )
        )

for block, name in enumerate(blocks):
    ax_heat.axvline(
        block * 5 - 0.5,
        color="white",
        lw=2.5,
        path_effects=[pe.withStroke(linewidth=5.0, foreground="#4d4d4d")],
    )
    ax_heat.text(
        min(block * 5 + 2, saturation.shape[1] - 1),
        -0.8,
        name,
        ha="center",
        va="bottom",
        fontsize=8.5,
    )

ax_heat.set_yticks(np.arange(len(heat_rows)))
ax_heat.set_yticklabels([labels.get(tag, tag) for tag in heat_rows])
for row, tag in enumerate(heat_rows):
    if not histories[tag]["feasible"]:
        ax_heat.get_yticklabels()[row].set_color(RED)
ax_heat.set_ylabel("Share of world carbon budget (%)", fontsize=COST_LABEL_SIZE)
ax_heat.set_xlabel("Constraint enforcement year", fontsize=COST_LABEL_SIZE)
ax_heat.set_title(
    "Binding constraints", fontsize=COST_LABEL_SIZE, loc="left", pad=34, fontweight="bold"
)

ACTIVE_CELL = GREYS(1.0 / 1.25)


def on_cell(mark):
    """One key: the mark drawn over the shade the cell it lands on actually has."""
    return (
        mpatches.Patch(facecolor=ACTIVE_CELL, edgecolor="none"),
        mlines.Line2D([], [], ls="none", **mark),
    )


ax_heat.legend(
    [
        on_cell(ACTIVE_MARK),
        on_cell(VIOLATED_MARK),
        mpatches.Patch(facecolor="white", edgecolor=RED, hatch="///"),
    ],
    ["Active", "Violated", "No feasible solution"],
    # ndivide=1 draws both artists over the whole handle, so the mark lands on the
    # patch; the default, None, splits the handle and sets them side by side.
    handler_map={tuple: HandlerTuple(ndivide=1)},
    # Wide enough that the mark sits inside its patch rather than overhanging it.
    handlelength=3.2,
    ncol=3,
    fontsize=SECONDARY_SIZE,
    frameon=False,
    loc="lower right",
    # Above the block names, which stand off the top of the grid on two lines.
    bbox_to_anchor=(1.0, 1.16),
)


def budget_colourbar(figure, rect):
    """The same budget axis as the figure above: gradient, a tick per scenario, references."""
    bar = figure.colorbar(
        plt.cm.ScalarMappable(cmap=budget_cmap, norm=budget_norm), cax=figure.add_axes(rect)
    )
    if bau is not None:
        bar.ax.set_ylim(BAR_MIN, bau_budget + 0.1)
    bar.set_label("Share of world carbon budget (%)", labelpad=15, fontsize=12)
    bar.set_ticks(tick_values)
    bar.set_ticklabels([labels[t] for t in sorted_tags])
    bar.ax.tick_params(length=0, labelsize=SECONDARY_SIZE)
    for value in tick_values:
        r, g, b, _ = budget_cmap(budget_norm(value))
        luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
        bar.ax.axhline(value, color="white" if luminance < 0.5 else "#222222", lw=0.8)

    def put(value, text, colour, ls, lw, offset=0.0):
        bar.ax.axhline(value, color=colour, lw=lw, ls=ls)
        bar.ax.text(
            LABEL_COLUMN,
            value + offset,
            text,
            color=colour,
            fontsize=SECONDARY_SIZE,
            va="center",
            ha="left",
            transform=bar.ax.get_yaxis_transform(),
        )

    if refueleu is not None:
        put(refueleu_budget, "ReFuelEU", RED, "--", 1.8, 0.015)
    if optimum is not None:
        put(optimum_budget, "Ref. optim.", GREEN, (0, (2, 4)), 2.0, -0.035)
    if bau is not None:
        put(bau_budget, "BAU", DARK_GREY, ":", 2.2)
    return bar


fig2.subplots_adjust(left=0.07, right=0.85, top=0.95, bottom=0.06, hspace=0.55, wspace=0.75)

# Each bar sits beside what it describes. The budget bar spans the two rows of curves it
# colours and stops there: it says nothing about the heatmap, which is on its own scale.
curves_top = ax_bio.get_position().y1
curves_bottom = ax_co2.get_position().y0
cbar2 = budget_colourbar(fig2, [0.885, curves_bottom, 0.015, curves_top - curves_bottom])

heat_box = ax_heat.get_position()
sat_bar = fig2.colorbar(heat, cax=fig2.add_axes([0.885, heat_box.y0, 0.015, heat_box.height]))
sat_bar.set_label("Normalised constraint use", fontsize=SMALL_LABEL_SIZE)
sat_bar.ax.tick_params(labelsize=SECONDARY_SIZE)

fig2.savefig("overall_layout2.pdf", bbox_inches="tight")

## 3. Which constraints bind, and when

No new model run: the constraint values at each optimum are already in the HDF that
`01_optimisation_runs.ipynb` saves beside every result. Five constraints, each enforced
at the five ReFuelEU reference years, plus the scalar carbon budget — 26 numbers per
run, negative for slack, zero for active, positive for violated.

The first figure is those values as they are written; the second is their saturation,
the share of each limit the run actually uses, with a dash where a constraint sits
exactly on its limit and a red dash where it is past it. The right-hand panel is the
mandate that produced them, on the same scale read as a percentage.

The ramp-up constraints are normalised by their volume cap, so a pathway using none of
its allowance sits at exactly −1, i.e. zero saturation; the first figure is clipped
there.

In [ ]:
ROWS = [R.budget_tag(b) for b in sorted(BUDGETS, reverse=True)] + ["mincarb"]

constraints = {}
for tag in ROWS:
    hdf = R.RESULTS_DIR / f"opt_main_{tag}.hdf"
    read = R.read_constraints(hdf) if hdf.exists() else None
    if read is not None:
        constraints[tag] = read
missing = [tag for tag in ROWS if tag not in constraints]
if missing:
    print("no history on disk yet ->", ", ".join(missing))

shown = [tag for tag in ROWS if tag in constraints]

# A pathway held near zero has consumed none of its allowed ramp, which puts its
# constraint at exactly -1: the floor is an artefact of the normalisation, not a run
# that is unusually comfortable.
FLOOR = -1.05

cmap = plt.get_cmap("Blues")
budget_tags = [tag for tag in shown if tag != "mincarb"]
colours = dict(zip(budget_tags, cmap(np.linspace(0.35, 1.0, len(budget_tags)))))
colours["mincarb"] = GREEN

fig, axes = plt.subplots(1, len(R.CONSTRAINT_LABELS), figsize=(17, 3.8), sharey=True)
for ax, (name, pretty) in zip(axes, R.CONSTRAINT_LABELS.items()):
    for tag in shown:
        run = constraints[tag]
        if name in run["constraints"]:
            # A budget with no feasible solution is drawn dashed: the point plotted is
            # the optimiser's last iterate, not an optimum.
            ax.plot(
                run["constraints"].index,
                np.clip(run["constraints"][name], FLOOR + 0.02, None),
                "o-" if run["feasible"] else "o--",
                ms=3,
                lw=2.0 if tag == "mincarb" else 1.5,
                color=colours[tag],
                label=labels.get(tag, tag) + ("" if run["feasible"] else " (infeasible)"),
            )
    ax.axhline(0, color=RED, lw=1.2)
    ax.axhspan(FLOOR, 0, color="#f4f4f4", zorder=0)
    ax.set_title(pretty.replace("\n", " "), fontsize=10)
    ax.set_xticks(R.OPTIM_YEARS)
    ax.set_xlabel("Enforcement year")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Constraint value (clipped at -1)")
axes[0].set_ylim(FLOOR, 0.25)
handles, names = axes[0].get_legend_handles_labels()
fig.legend(
    handles, names, title="Budget (Gt)", fontsize=8, loc="center left", bbox_to_anchor=(1.0, 0.5)
)
tidy(axes)
plt.tight_layout()
plt.savefig("constraints_by_budget.pdf", bbox_inches="tight")

In [ ]:
# Saturation: the share of each limit the run uses. Every constraint is written as
# (use - limit) / limit, so 1 + g is that share directly, and 1.0 is the limit itself.
# The mandate is a share already, on its own 0-100 % scale.
saturation = np.full((len(shown), len(R.CONSTRAINT_LABELS) * len(R.OPTIM_YEARS) + 1), np.nan)
mandate = np.full((len(shown), 2 * len(R.OPTIM_YEARS)), np.nan)
for row, tag in enumerate(shown):
    run = constraints[tag]
    for block, name in enumerate(R.CONSTRAINT_LABELS):
        if name in run["constraints"]:
            saturation[row, block * 5 : block * 5 + 5] = 1 + run["constraints"][name].values
    saturation[row, -1] = 1 + run["carbon"]
    mandate[row, :5] = run["mandate"]["biofuel"].values
    mandate[row, 5:] = run["mandate"]["electrofuel"].values

ACTIVE = 1e-3
years = [str(year) for year in R.OPTIM_YEARS]
blocks = [pretty for pretty in R.CONSTRAINT_LABELS.values()] + ["Carbon\nbudget"]

fig, (ax_g, ax_x) = plt.subplots(
    1, 2, figsize=(17, 4.4), sharey=True, gridspec_kw={"width_ratios": [26, 10], "wspace": 0.04}
)

# Left: how close every constraint sits to its limit, saturated blue at the limit.
image = ax_g.imshow(saturation, aspect="auto", cmap="Blues", vmin=0, vmax=1.25)
for row in range(saturation.shape[0]):
    for column in range(saturation.shape[1]):
        value = saturation[row, column]
        if np.isnan(value):
            continue
        if value > 1 + ACTIVE:
            ax_g.plot(column, row, marker="_", color=RED, ms=9, mew=2.4)
        elif value >= 1 - ACTIVE:
            ax_g.plot(column, row, marker="_", color="black", ms=9, mew=2)

# Right: the mandate that produced them, on its own scale.
image_x = ax_x.imshow(mandate, aspect="auto", cmap="YlGn", vmin=0, vmax=100)

for ax, columns, names in [
    (ax_g, saturation.shape[1], years * len(R.CONSTRAINT_LABELS) + ["all"]),
    (ax_x, mandate.shape[1], years * 2),
]:
    ax.set_xticks(np.arange(columns))
    ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_xticks(np.arange(columns + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(len(shown) + 1) - 0.5, minor=True)
    ax.grid(which="minor", color="white", lw=0.8)
    ax.tick_params(which="minor", length=0)
    for row, tag in enumerate(shown):
        # A run that ended infeasible has no optimum to read: mark the whole row.
        if not constraints[tag]["feasible"]:
            ax.add_patch(
                plt.Rectangle(
                    (-0.5, row - 0.5), columns, 1, fill=False, hatch="///", edgecolor=RED, lw=0
                )
            )

for block, name in enumerate(blocks):
    ax_g.axvline(block * 5 - 0.5, color="white", lw=3.5)
    ax_g.text(
        min(block * 5 + 2, saturation.shape[1] - 1),
        -0.8,
        name,
        ha="center",
        va="bottom",
        fontsize=8.5,
    )
for block, name in enumerate(["Biofuel\nmandate", "Electrofuel\nmandate"]):
    ax_x.axvline(block * 5 - 0.5, color="white", lw=3.5)
    ax_x.text(block * 5 + 2, -0.8, name, ha="center", va="bottom", fontsize=8.5)

ax_g.set_yticks(np.arange(len(shown)))
ax_g.set_yticklabels([labels.get(tag, tag) for tag in shown])
for row, tag in enumerate(shown):
    if not constraints[tag]["feasible"]:
        ax_g.get_yticklabels()[row].set_color(RED)
ax_g.set_ylabel("Carbon budget (Gt)")
ax_g.set_xlabel("Constraint enforcement year")
ax_x.set_xlabel("Reference year")

fig.colorbar(image, ax=ax_x, pad=0.02).set_label("Share of the limit used")
fig.colorbar(image_x, ax=ax_x, pad=0.02).set_label("Blending mandate (%)")

fig.legend(
    handles=[
        mlines.Line2D([], [], color="black", marker="_", ls="none", ms=9, mew=2, label="Active"),
        mlines.Line2D([], [], color=RED, marker="_", ls="none", ms=9, mew=2.4, label="Violated"),
        mpatches.Patch(facecolor="white", edgecolor=RED, hatch="///", label="No feasible solution"),
    ],
    ncol=3,
    fontsize=9,
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.45, 1.02),
)
plt.savefig("constraints_active_set.pdf", bbox_inches="tight")

The active set walks *backwards in time* as the budget tightens. At 3.8 only the biofuel
ramp-up binds, in 2045–2050; its binding years then step back a period at a time —
2040–2045 at 3.6, 2035–2040 at 3.4 — and from 3.0 down they are 2030–2035,
i.e. the scenario stops being limited by how much biomass exists and starts being
limited by how fast the plants can be built. At 2.0 Gt nothing satisfies the budget at
all — the row is hatched, and the violated constraint is the budget itself.

## 4. Biomass allocation — the trade-off surface

Figure 9. Every run of every biomass case is one point in (CO₂ consumed, biomass
allocated); the surface interpolated through them is the cost of the pair. The fossil
BAU appears at each allocation because it uses no biomass at all.

ReFuelEU appears twice, as in the published figure: **linear**, the mandate interpolated
between the regulation's milestones, and **step**, each milestone held flat until the
next — which is what the regulation actually obliges. Step abates less early, so it
consumes more of the budget (3.30 % against 3.12 % of the world budget) at a lower
surplus loss.

The reference case now sits at 10 % rather than 9.90 %. Each ReFuelEU point is labelled
with its own run's surplus loss and, alongside it, the delta against the surface read at
that exact (CO₂, biomass) position — how far the actual run sits from what the
interpolated cost surface would predict there.

In [ ]:
BIOMASS_CASES = {"B5": 5.0, "B75": 7.5, "main": 10.0, "B15": 15.0}
CASE_COLOURS = {"main": "#cb3629", "B5": "#092054", "B15": "#efbd40", "B75": "#7e9b59"}

# At the tightest budgets, some (case, budget) continuations never reach the target:
# the biofuel/electrofuel ramp-up constraints cap how fast the fleet can decarbonise,
# so the optimiser lands on the best it can do with the emissions constraint still
# violated. `aviation_carbon_budget_constraint` is (actual - allowed) / allowed, so
# positive means over budget; worst at low biomass (B5, B75) and the tightest targets,
# where there is the least alternative fuel to lean on. These are not points on the
# (CO2, biomass) cost surface -- the CO2 axis is not the one that was asked for -- so
# they are dropped rather than plotted as if the trade-off surface passed through them.
CONSTRAINT_TOL = 0.01

points = []
excluded = []
for case, biomass in BIOMASS_CASES.items():
    for tag, (vector, floats) in collect(case, ALL_TAGS).items():
        violation = floats.get("aviation_carbon_budget_constraint", 0.0)
        if tag != "mincarb" and violation > CONSTRAINT_TOL:
            excluded.append((case, tag, violation))
            continue
        points.append(
            {
                "case": case,
                "biomass": biomass,
                "co2": floats["carbon_budget_consumed_share"] / R.EU_ASK_SHARE,
                "surplus": vector["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9,
            }
        )

if excluded:
    print(
        f"Dropped {len(excluded)} infeasible run(s) (budget constraint violated beyond {CONSTRAINT_TOL:.0%}):"
    )
    for case, tag, violation in excluded:
        print(f"  {case} {tag}: {violation:+.1%} over its allocated budget")

# The BAU is biomass-free, so it is the same scenario at every allocation.
if available("main", "fossil"):
    bau_vector, bau_floats = R.load(R.RESULTS_DIR / "fossil_main.json")
    for biomass in BIOMASS_CASES.values():
        points.append(
            {
                "case": "fossil",
                "biomass": biomass,
                "co2": bau_floats["carbon_budget_consumed_share"] / R.EU_ASK_SHARE,
                "surplus": bau_vector["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9,
            }
        )

points = pd.DataFrame(points)
print(f"{len(points)} scenario points")
points.head()

In [ ]:
# Pad the interpolation domain a bit beyond the scenario points themselves, so the
# surface does not run flush to the axes and the edge points get some breathing room.
# griddata leaves the padded margin NaN (outside the convex hull of the data), which
# pcolormesh simply leaves uncoloured -- a clean border around the coloured surface.
pad_x = 0.06 * (points["co2"].max() - points["co2"].min())
pad_y = 0.06 * (points["biomass"].max() - points["biomass"].min())
grid_x, grid_y = np.meshgrid(
    np.linspace(points["co2"].min() - pad_x, points["co2"].max() + pad_x, 200),
    np.linspace(points["biomass"].min() - pad_y, points["biomass"].max() + pad_y, 200),
)
grid_z = griddata(
    (points["co2"], points["biomass"]), points["surplus"], (grid_x, grid_y), method="cubic"
)

# Diverging scale centred on zero, so the sign of the surplus change is readable.
limit = 270
cmap = plt.get_cmap("RdBu_r")
norm = mcolors.Normalize(vmin=-limit, vmax=limit)

plt.figure(figsize=(10, 7))
mesh = plt.pcolormesh(grid_x, grid_y, grid_z, shading="gouraud", cmap=cmap, norm=norm)
contours = plt.contour(grid_x, grid_y, grid_z, levels=15, colors="black", linewidths=0.8)
for label in plt.clabel(contours, inline=True, fontsize=11, fmt="%.1f", colors="black"):
    label.set_path_effects([pe.withStroke(linewidth=3, foreground="white")])

plt.colorbar(mesh).set_label("Cumulative total surplus loss (Bn€, discounted)")
plt.scatter(
    points["co2"],
    points["biomass"],
    color=GREY,
    edgecolor="black",
    s=35,
    alpha=0.5,
    label="Scenario point",
    zorder=5,
)

# Both readings of the regulation, as in the published figure. "Linear" interpolates the
# mandate between ReFuelEU's milestones; "step" holds each milestone flat until the next,
# which is what the regulation actually obliges. The step run is the preflight's MDA: it
# is not part of block E, but it is built in the same configuration -- the preflight's
# linear run is identical to block E's refueleu_main on every input and to 6e-05 EUR on
# the objective.
REFUELEU_STEP_PATH = Path("sweep/results/refueleu_step.json")
refueleu_step = R.load(REFUELEU_STEP_PATH) if REFUELEU_STEP_PATH.exists() else None
if refueleu_step is None:
    print("ReFuelEU (step) not on disk yet -> run sweep/preflight.py")

# (run, label, text offset, text alignment): linear labelled above-left, step below-right.
for run, name, (dx, dy), (ha, va) in [
    (refueleu, "ReFuelEU (linear)", (-0.06, 0.7), ("right", "bottom")),
    (refueleu_step, "ReFuelEU (step)", (0.06, -0.7), ("left", "top")),
]:
    if run is None:
        continue
    surplus = run[0]["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9
    co2 = run[1]["carbon_budget_consumed_share"] / R.EU_ASK_SHARE
    # The regulation implies its own biomass draw rather than choosing one; read it off
    # the run instead of placing the point by hand.
    biomass = run[0]["generic_biomass_consumed_global_share"].loc[2050] / R.EU_ASK_SHARE
    # Read the interpolated surface at that exact (co2, biomass) position, rather than
    # marking the ReFuelEU-equivalent budget with its own vertical line -- the surface
    # already encodes what a scenario at that spot costs, biomass draw included.
    surface = griddata(
        (points["co2"], points["biomass"]), points["surplus"], (co2, biomass), method="cubic"
    )
    delta = surplus - float(surface)
    plt.scatter(
        co2,
        biomass,
        s=100,
        color=cmap(norm(surplus)),
        edgecolor="black",
        lw=1.2,
        zorder=10,
        label="ReFuelEU" if name.endswith("(linear)") else None,
        path_effects=[pe.withStroke(linewidth=4, foreground="white")],
    )
    name_bold = name.replace(" ", r"\ ")  # mathtext bold drops literal spaces
    plt.text(
        co2 + dx,
        biomass + dy,
        f"$\\bf{{{name_bold}}}$\n"
        f"\u0394TS {surplus:+.1f} Bn€\n"
        f"\u0394 vs surface {delta:+.1f} Bn€",
        fontsize=10.5,
        linespacing=1.8,
        color="purple",
        ha=ha,
        va=va,
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            edgecolor="purple",
            linewidth=0.8,
            alpha=0.45,
        ),
        zorder=11,
    )

plt.xlabel("Carbon budget consumed share (%)")
plt.ylabel("Biomass allocated to aviation (%)")
plt.grid(True, ls="--")
plt.legend()
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("2d_shares.pdf")

## 5. Pessimistic technology roadmap

Figure 10 and §4.3. Same problem with the drop-in efficiency gain cut from 1.35 to
0.91 %/yr — the only difference between `pess` and `main`. The dashed line is the extra
surplus loss attributable to the missing efficiency at each budget, which is the number
the section is really about.

In [ ]:
curves = {}
for case in ["main", "pess"]:
    rows = [
        {
            "co2": floats["carbon_budget_consumed_share"] / R.EU_ASK_SHARE,
            "surplus": vector["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9,
            "tag": tag,
        }
        for tag, (vector, floats) in collect(case, ALL_TAGS).items()
    ]
    if available(case, "fossil"):
        vector, floats = R.load(R.RESULTS_DIR / f"fossil_{case}.json")
        rows.append(
            {
                "co2": floats["carbon_budget_consumed_share"] / R.EU_ASK_SHARE,
                "surplus": vector["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9,
                "tag": "fossil",
            }
        )
    curves[case] = pd.DataFrame(rows).sort_values("co2")

plt.figure(figsize=(10, 6))
for case, colour, label in [("main", RED, "Reference"), ("pess", BLUE, "Low efficiency")]:
    plt.plot(curves[case]["co2"], curves[case]["surplus"], "-o", color=colour, label=label)
    for _, row in curves[case].iterrows():
        if row["tag"] in ("mincarb", "fossil"):
            plt.annotate(
                {"mincarb": "Min $CO_2$", "fossil": "Fossil"}[row["tag"]],
                (row["co2"], row["surplus"]),
                textcoords="offset points",
                xytext=(5, 5),
                ha="left",
            )

# Difference at matched CO2, extrapolating flat outside the low-efficiency range.
common = curves["main"]["co2"].values
delta = (
    np.interp(common, curves["pess"]["co2"], curves["pess"]["surplus"])
    - curves["main"]["surplus"].values
)
plt.plot(common[2:], delta[2:], "--o", color="black", label="Relative difference")

plt.xlabel("Carbon budget consumed share (%)")
plt.ylabel("Total and relative surplus loss (Bn€, discounted)")
plt.grid(True)
plt.legend()
tidy(plt.gca())
plt.tight_layout()
plt.savefig("technology.pdf")

In [ ]:
# Where the difference lands: on airlines, on passengers' wallets, or on traffic.
ref, _ = R.load(R.RESULTS_DIR / "opt_main_2_8.json")
low, _ = R.load(R.RESULTS_DIR / "opt_pess_2_8.json")

AIRFARE_2019 = 0.09236379319842411


def surplus_loss(v):
    """Annual passenger surplus loss relative to the 2019 price, in Bn€.

    The stored `area_loss` is the integral under the demand curve; the two correction
    terms move it from the counterfactual traffic to the realised one.
    """
    return (
        v["area_loss"]
        - AIRFARE_2019 * (v["rpk_no_elasticity"] - v["rpk"])
        + v["rpk"] * (v["airfare_per_rpk"] - AIRFARE_2019)
    ) / 1e9


panels = [
    (
        "Airline revenue (Bn€)",
        lambda v: (v["airfare_per_rpk"] - v["total_cost_per_rpk"]) * v["rpk"] / 1e9,
    ),
    ("Passenger spending (Bn€)", lambda v: v["airfare_per_rpk"] * v["rpk"] / 1e9),
    ("Passenger surplus loss (Bn€)", surplus_loss),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (title, metric) in zip(axes, panels):
    reference, degraded = metric(ref), metric(low)
    ax.plot(YEARS, reference, color=BLUE, label="Reference roadmap")
    ax.plot(YEARS, degraded, color=RED, label="Low efficiency")
    ax.fill_between(YEARS, reference, degraded, alpha=0.1, color=RED, label="Difference")
    cumulative = np.nansum(reference - degraded)
    ax.set_title(f"{title}\ncumulative Δ = {cumulative:.1f} Bn€")
    ax.set_xlabel("Year")
    ax.set_xlim(2020, 2050)
    ax.grid(True)
    ax.legend(fontsize=9)

tidy(axes)
plt.tight_layout()

## 6. Electrofuel abatement cost against the discount rate

Figure 11 — the only figure that needs no optimisation at all, since it compares the
pathways' own costs and emission factors. The surface is the electrofuel cost of
abatement discounted back to 2020; the dot-dashed lines mark where biofuel entering
service in a given year becomes the cheaper option.

Both quantities come from the *pathway* outputs, so this is a property of
`energy_rte.yaml` rather than of any run — a useful check that the carriers were
migrated correctly.

In [ ]:
prices = R.load(R.RESULTS_DIR / "refueleu_main.json")[0]

# MFSP is EUR/MJ on main; emission factors gCO2/MJ -> tCO2/MJ.
delta_price = {
    "biofuel": (prices["generic_biofuel_mean_mfsp"] - prices["fossil_kerosene_mean_mfsp"]).loc[
        2020:2050
    ],
    "electrofuel": (
        prices["generic_electrofuel_mean_mfsp"] - prices["fossil_kerosene_mean_mfsp"]
    ).loc[2020:2050],
}
delta_emissions = {
    "biofuel": (
        (
            prices["fossil_kerosene_mean_co2_emission_factor"]
            - prices["generic_biofuel_mean_co2_emission_factor"]
        )
        / 1e6
    ).loc[2020:2050],
    "electrofuel": (
        (
            prices["fossil_kerosene_mean_co2_emission_factor"]
            - prices["generic_electrofuel_mean_co2_emission_factor"]
        )
        / 1e6
    ).loc[2020:2050],
}

years_fine = np.arange(2020, 2050.05, 0.01)
rates = np.linspace(0.0, 0.2, 4000)
years_int = np.arange(2020, 2051)


def interpolate(series):
    return np.interp(years_fine, years_int, series.values)


# Electrofuel CAC, discounted from its entry-into-service year back to 2020.
cac = interpolate(delta_price["electrofuel"])[:, None] / (
    interpolate(delta_emissions["electrofuel"])[:, None]
    * (1 + rates[None, :]) ** (years_fine - 2020)[:, None]
)

plt.figure(figsize=(10, 5))
image = plt.imshow(
    np.clip(cac, 0, None),
    aspect="auto",
    origin="lower",
    extent=[0, 20, years_fine[0], years_fine[-1]],
    cmap="RdBu_r",
    interpolation="nearest",
    vmin=0,
    vmax=1500,
)

# Where electrofuel is *more* emissive than fossil the ratio is meaningless.
plt.contourf(
    rates * 100, years_fine, cac, levels=[cac.min(), 0], colors=RED, alpha=0.5, hatches=["///"]
)
plt.text(
    7,
    2023,
    "E-fuel more emissive than fossil",
    color=RED,
    fontsize=13,
    bbox=dict(facecolor="white", edgecolor=RED, boxstyle="round,pad=0.3"),
)

for entry_year in [2025, 2030, 2035, 2040]:
    threshold = np.interp(entry_year, years_int, delta_price["biofuel"]) / (
        np.interp(entry_year, years_int, delta_emissions["biofuel"])
        * (1 + rates) ** (entry_year - 2020)
    )
    crossing = np.full_like(rates, np.nan, dtype=float)
    for j in range(len(rates)):
        below = np.where((cac[:, j] <= threshold[j]) & (cac[:, j] >= 0))[0]
        if below.size:
            crossing[j] = years_fine[below[0]]
    plt.plot(rates * 100, crossing, color="white", lw=0.8, ls="-.")
    valid = np.where(~np.isnan(crossing))[0]
    if valid.size:
        plt.text(
            rates[valid[-1]] * 100 - 1.8,
            crossing[valid[-1]] + 0.9,
            f"EIS: {entry_year}",
            color="white",
            fontsize=9,
            va="center",
            rotation=-5,
        )

masked = np.ma.masked_where((cac <= 0) | (cac >= 1600), cac)
# Only the widely-spaced levels carry labels: the CAC blows up as the emission benefit
# goes to zero, so every contour above ~600 is pinched into the same corner and their
# labels land on top of each other.
labelled = [100, 200, 300, 400, 600]
iso = plt.contour(
    rates * 100,
    years_fine,
    masked,
    levels=labelled,
    colors="dimgrey",
    linewidths=0.8,
    linestyles="--",
)
for label in plt.clabel(iso, inline=True, fontsize=8, fmt=lambda x: f"{int(x)}"):
    label.set_color("white")
    label.set_path_effects([pe.withStroke(linewidth=1, foreground="dimgrey")])
plt.contour(
    rates * 100,
    years_fine,
    masked,
    levels=[lvl for lvl in np.arange(100, 1600, 100) if lvl not in labelled],
    colors="lightgrey",
    linewidths=0.5,
    linestyles=":",
)

plt.plot([], [], color="dimgrey", lw=0.8, ls="--", label="Electrofuel iso-CAC")
plt.plot([], [], color="white", lw=0.8, ls="-.", label="Below: biofuel is cheaper")
plt.legend(loc="upper right", fontsize=10, facecolor="white", edgecolor="white")

plt.colorbar(image).set_label(r"Electrofuel CAC (€/tCO$_2$), discounted to 2020")
plt.xlabel("Discount rate (%)")
plt.ylabel("Entry into service")
plt.tight_layout()
plt.savefig("sensitivity.pdf")

### 6b. The same costs as curves

The surface above is hard to read a number off: it asks the eye to turn a colour into a
euro figure, and the biofuel comparison is buried in the dot-dashed contours. The same
two quantities plot conventionally — the abatement cost against the discount rate, one
curve per entry-into-service year — and the pathways then separate by line style rather
than by contour.

That also makes room for block C's **dedicated-wind electrofuel** as a third pathway. It
is carried here as an extra curve, not as a replacement for the grid-electricity one:
both are properties of the carriers, so the two sit on the same axes and the gap between
them is the value of building dedicated generation.

Two things the surface does not say out loud. Within one entry-into-service year the
three pathways are *parallel* on the log axis — they carry the same discount factor
`(1+r)^(EIS-2020)`, so the ranking between pathways at a fixed year does not depend on
the discount rate at all; only comparisons *across* years are re-ordered by it. And grid
electrofuel entering in 2025 has no curve, for the reason the hatched corner of the
surface gives: it is still more emissive than fossil kerosene then, and only crosses
over in 2028.

In [ ]:
# --- Section 6, read as curves rather than as a surface -------------------------------
# The same quantity as the image above, taken along the discount-rate axis instead of
# across it: one curve per entry-into-service year, one line style per pathway. Block
# C's dedicated-wind electrofuel joins as a third pathway rather than replacing the grid
# one -- its prices and emission factors are a property of the carriers swapped into
# `efuel_wind.json`, not of that run's optimum, so it belongs on these axes too.
wind_prices = R.load(Path("sweep/results") / "efuel_wind.json")[0]

# Colour is the pathway, matching its meaning everywhere else in this notebook (GREEN
# for biofuel, BLUE for electrofuel); dedicated wind is electrofuel's own carriers on a
# different generation source, so it gets a third, clearly separate hue rather than a
# shade of BLUE.
WIND = "#7b3294"
PATHWAYS = {
    "Biofuel": ("generic_biofuel", prices, GREEN),
    "Electrofuel (grid electricity)": ("generic_electrofuel", prices, BLUE),
    "Electrofuel (dedicated wind)": ("generic_electrofuel", wind_prices, WIND),
}
EIS_YEARS = [2025, 2030, 2035, 2040]
EIS_STYLE = dict(zip(EIS_YEARS, ["-", "--", "-.", ":"]))
BASELINE_RATE = 4.5

rates = np.linspace(0.0, 0.2, 400)


def abatement_cost(carrier, source, entry_year):
    """CAC of `carrier` entering service in `entry_year`, discounted back to 2020.

    None where the pathway is no cleaner than fossil that year: the ratio has no
    meaning there, and grid electrofuel entering in 2025 is exactly that case.
    """
    delta_price = (
        source[f"{carrier}_mean_mfsp"].loc[entry_year]
        - prices["fossil_kerosene_mean_mfsp"].loc[entry_year]
    )
    delta_emissions = (
        prices["fossil_kerosene_mean_co2_emission_factor"].loc[entry_year]
        - source[f"{carrier}_mean_co2_emission_factor"].loc[entry_year]
    ) / 1e6
    if delta_emissions <= 0:
        return None
    return delta_price / (delta_emissions * (1 + rates) ** (entry_year - 2020))


figure, axis = plt.subplots(figsize=(9.5, 5.5))

axis.axvline(BASELINE_RATE, color="0.75", lw=1, ls="-", zorder=0)
axis.text(
    BASELINE_RATE + 0.2,
    2900,
    f"baseline {BASELINE_RATE} %",
    color="0.45",
    fontsize=SECONDARY_SIZE,
    va="top",
)

for label, (carrier, source, colour) in PATHWAYS.items():
    for entry_year in EIS_YEARS:
        cac = abatement_cost(carrier, source, entry_year)
        if cac is None:
            continue
        axis.plot(rates * 100, cac, color=colour, ls=EIS_STYLE[entry_year], lw=2.0)

axis.set_yscale("log")
axis.set_xlim(0, 20)
axis.set_ylim(8, 3200)
# Decades alone are too coarse a rule on three decades of log axis: the interesting
# range is the 100-1000 band and it would carry no tick at all.
axis.set_yticks([10, 20, 50, 100, 200, 500, 1000, 2000])
axis.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
axis.set_yticks([], minor=True)
axis.set_xlabel("Social discount rate (%)")
axis.set_ylabel(r"Cost of abatement (€/tCO$_2$), discounted to 2020")
tidy(axis)

# Two legends, colour and line style read separately -- both corners they sit in are
# empty at every rate, since no curve is ever both cheap and undiscounted at once.
pathway_legend = axis.legend(
    handles=[
        mlines.Line2D([], [], color=colour, lw=2.2, label=label)
        for label, (_, _, colour) in PATHWAYS.items()
    ],
    title="pathway",
    loc="upper right",
    frameon=False,
    title_fontsize=SECONDARY_SIZE,
)
axis.add_artist(pathway_legend)
axis.legend(
    handles=[
        mlines.Line2D([], [], color="0.35", ls=style, lw=2.0, label=f"EIS {year}")
        for year, style in EIS_STYLE.items()
    ],
    title="entry into service",
    loc="lower left",
    frameon=False,
    title_fontsize=SECONDARY_SIZE,
)

plt.tight_layout()
plt.savefig("sensitivity_curves.pdf")

### 6c. The transpose: cost against entry into service

6b fixed the entry-into-service year and read the cost across discount rates; this
fixes the discount rate and reads it across entry-into-service years instead, at the
four rates section 7.4 already uses for block D — 3.2 % and 7 % either side of the
4.5 % baseline, plus the 15 % extreme that breaks the optimiser's vertex.

The near-vertical rise each electrofuel curve makes around 2028-2030 is the same
feature as the hatched corner of the surface, seen from the other side: just after grid
electrofuel turns cleaner than fossil kerosene, the emissions saving is still close to
zero, so a finite price gap divided by it sends the cost of abatement to arbitrarily
large numbers before it settles down as the gap widens. Dedicated wind, cleaner than
fossil throughout, has no such corner to round.

In [ ]:
# --- Section 6c, the transpose: CAC against entry into service, discrete rates --------
# 6b fixed the entry-into-service year and swept the discount rate; this fixes the
# discount rate and sweeps entry into service instead, at the same four rates block D
# probes in section 7.4 -- 3.2 % and 7 % either side of the 4.5 % baseline, plus 15 % as
# the extreme that breaks the optimiser's vertex. `PATHWAYS` and `wind_prices` are 6b's.
DISCOUNT_RATES = [0.032, 0.045, 0.07, 0.15]
RATE_LABEL = {0.032: "3.2 %", 0.045: "4.5 % (baseline)", 0.07: "7 %", 0.15: "15 %"}
RATE_STYLE = dict(zip(DISCOUNT_RATES, ["-", "--", "-.", ":"]))

eis_years = np.arange(2020, 2051)


def cac_vs_eis(carrier, source, rate):
    """CAC of `carrier` by entry-into-service year, discounted to 2020 at `rate`.

    NaN wherever the pathway is not yet cleaner than fossil that year.
    """
    delta_price = (
        source[f"{carrier}_mean_mfsp"].loc[eis_years]
        - prices["fossil_kerosene_mean_mfsp"].loc[eis_years]
    )
    delta_emissions = (
        prices["fossil_kerosene_mean_co2_emission_factor"].loc[eis_years]
        - source[f"{carrier}_mean_co2_emission_factor"].loc[eis_years]
    ) / 1e6
    cac = delta_price / (delta_emissions * (1 + rate) ** (eis_years - 2020))
    return cac.where(delta_emissions > 0)


figure, axis = plt.subplots(figsize=(9.5, 5.5))

for label, (carrier, source, colour) in PATHWAYS.items():
    for rate in DISCOUNT_RATES:
        cac = cac_vs_eis(carrier, source, rate).dropna()
        axis.plot(cac.index, cac, color=colour, ls=RATE_STYLE[rate], lw=2.0)

axis.set_yscale("log")
axis.set_xlim(2020, 2050)
axis.set_ylim(5, 3200)
axis.set_yticks([5, 10, 20, 50, 100, 200, 500, 1000, 2000])
axis.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
axis.set_yticks([], minor=True)
axis.set_xlabel("Entry into service")
axis.set_ylabel(r"Cost of abatement (€/tCO$_2$), discounted to 2020")
tidy(axis)

# Two legends, colour and line style read separately -- same split as 6b, and both
# corners are empty here too: nothing is ever both cheap and entering service early.
pathway_legend = axis.legend(
    handles=[
        mlines.Line2D([], [], color=colour, lw=2.2, label=label)
        for label, (_, _, colour) in PATHWAYS.items()
    ],
    title="pathway",
    loc="upper right",
    frameon=False,
    title_fontsize=SECONDARY_SIZE,
)
axis.add_artist(pathway_legend)
axis.legend(
    handles=[
        mlines.Line2D([], [], color="0.35", ls=style, lw=2.0, label=RATE_LABEL[rate])
        for rate, style in RATE_STYLE.items()
    ],
    title="discount rate",
    loc="lower left",
    frameon=False,
    title_fontsize=SECONDARY_SIZE,
)

plt.tight_layout()
plt.savefig("sensitivity_curves_by_eis.pdf")

## 7. One-parameter sensitivities — blocks A to D

A different experiment from everything above. Sections 1–6 sweep the carbon budget at
fixed parameters; this holds the budget fixed at the ReFuelEU-equivalent 3.8656 GtCO₂ and
varies one parameter at a time. Fourteen optimisations, described in
`01_optimisation_runs.ipynb` §3 and run from `sweep/`. The summary in 7.5 also draws on block E, placing
these one-parameter changes beside the carbon budget, the biomass allocation and the
technology roadmap.

All five figures are drawn by the scripts in `sweep/`, so the notebook and the command
line produce the same files.

Every time axis starts in **2019**, the last historical year and the state everything is
measured against: the airfare anchor, the objective's 2019 unit economics and the
frozen-2019 emissions baseline. So the COVID drop in 2020 shows as a drop, and every
elastic run's airfare starts exactly on its anchor. Fixed demand starts 0.3 % below it:
the no-feedback chain computes its own 2019 fare rather than taking the anchor, and its
cost objective never reads the fare.

Every time-series figure ends with the same cost row: **g** the airfare, in euro cents per
RPK, against its 2019 anchor; **h** the direct operating cost and **i** its energy
component, in euro cents per ASK, as the model computes them. The row is scaled to
2023–2050 because 2020 — load factor down to 65 % — lifts every per-seat and per-passenger
cost identically in every case; where that pushes 2020 off the axis its value is printed
at the edge.

**What the block reports, in one line each:**

* **A — price elasticity.** The required mandate *rises* with |ε|, 51.9 → 55.8 % from
  −0.6 to −1.4, which is the opposite of the obvious expectation. Fares sit below the 2019
  anchor until 2046–2048, so the demand response *adds* traffic for most of the horizon
  and only suppresses it at the end: cumulative RPK rises with |ε| while 2050 RPK falls.
  Against a **cumulative** budget the early traffic has to be paid for later.
* **A′ — fixed demand.** 49.0 %, the lowest of all, and the cheapest per tonne abated
  (−39 EUR/tCO₂ averaged over 2020–2050, against −30 for the baseline): with no demand
  response there is no forgone travel to pay for.
* **B / B′ — ramp-up limits.** 52.8 → 62.8 %. A tighter ramp cannot abate early, and
  against a cumulative budget that means it must end *higher*, not lower.
* **C — electrofuel pathway.** The largest mover short of the extreme discount rate:
  90.4 % of the blend against 53.2 %, electrofuel 49 % of it against 14 %, entering a
  period earlier. Discounted at the objective's own rate, late wind electrofuel undercuts
  *early* biofuel, so the optimiser holds 2030 biofuel at 2 % against 10 % and buys the
  abatement back after 2040. Policy cost falls from 147 to 134 bn €.
* **D — discount rate.** 3.2 % and 4.5 % are the same design to every digit; 7 % moves the
  mandate to 64.4 %; at 15 % it reaches 92.0 % of the blend and the reported policy cost
  collapses to 18.5 bn € while the realised cost per tonne *rises*.

Biofuel is biomass-capped in every run from 2045, at 39–44 % of the blend. Most runs reach
the cap by 2040; the ones that defer abatement — dedicated wind, 15 %, and the two tight
ramps — get there later. Past the cap, every adjustment the optimiser makes lands on
electrofuel.

**What the corrected ramp-up changed.** Before Eq. 12's volume branch was read as an
increment, dedicated wind gave the *same* 59.2 % mandate as the baseline and this block
reported that the pathway "barely registers". The absolute-ceiling reading had been
capping electrofuel's growth once it was established, so a cheaper pathway had nowhere to
go. The baseline itself moved from 59.2 to 53.2 %, because biofuel can now ramp harder in
2030. Runs whose volume branch never bound — the 39 %/yr rate, the 0.4 EJ/yr volume, and
15 % — did not move: the new budget is larger by exactly the 4.06 MtCO₂ the 2025 step
adds to 2021–2024, so those two changes cancel.

In [ ]:
import sys

sys.path.insert(0, str(Path("sweep").resolve()))

import importlib  # noqa: E402

import plot_elasticity  # noqa: E402
import plot_sensitivities as PS  # noqa: E402

# Re-importing a module in a live kernel returns the copy already loaded, so an edit to
# the plotting scripts would not show until a restart. Reload them instead -- the
# sensitivities module first, since plot_elasticity takes its layout from it.
PS = importlib.reload(PS)
plot_elasticity = importlib.reload(plot_elasticity)

summary = pd.read_csv("sweep/sweep_summary.csv")
summary[
    [
        "run",
        "block",
        "label",
        "feasible",
        "aaf_share_2050",
        "electrofuel_share_2050",
        "rpk_2050_Tpkm",
        "policy_cost_bnEUR",
    ]
].round(3)

### 7.1 Price elasticity

The inset carries 2044–2050 at its own scale: the spread is under 3 % of a trajectory
that grows fivefold, so on the full axis the five lines lie on top of one another.

Panel **c** is a realised *average* cost — the whole scenario over the whole abatement —
measured against the same counterfactual the objective uses: 2019 technology flying the
no-elasticity traffic. The numerator is `area_loss + total_airline_cost_increase`, the two
terms `cumulative_total_surplus_loss` sums, both already measured against that state. The
denominator is `co2_emissions_last_historical_year_technology_baseline3` minus what the
run emits; that series is `rpk_reference` — identical to `rpk_no_elasticity` — at frozen
2019 energy per ASK, load factor and emission factor, i.e. the emissions of the very same
reference state.

It is negative for most of the horizon, and that is the finding rather than a defect:
against a world that froze in 2019 the scenario is both cheaper and cleaner, because it
carries every efficiency and operational gain. It turns positive only near 2050, once the
mandate carries the abatement on its own. Measured against each run's matched fossil BAU
instead, the baseline reads 261–366 EUR/tCO₂. That answers a different question — what
the *mandate alone* costs, holding the efficiency gains fixed — and is not the objective's
basis.

In [ ]:
plot_elasticity.main()

### 7.2 Industrial ramp-up limits

Rate variants solid, volume variants dashed; warm is tighter than the baseline, cool is
looser — a diverging encoding, because these vary *around* a centre.

Panel **d** is the one to read: the loose variants push biofuel to 30–34 % by 2035 while
the tight ones are still at 15–19 %, and by 2045 all five have converged on the same
biomass ceiling. The ramp cap does not change where biofuel ends, only how fast it gets
there — and everything that could not be abated early reappears in panel **e** as
electrofuel.

In [ ]:
PS.ramp_up()

### 7.3 Electrofuel pathway

Top row is the inputs that were swapped, bottom row what came out. Panel **c** turns the
two into a cost per tonne abated against fossil kerosene, **discounted to 2020** at the
run's own rate while the tonnes are not. That asymmetry is the point: the objective is a
discounted sum of euros and G1 caps an undiscounted sum of tonnes, so this is the ratio the
optimiser actually ranks options by. Undiscounted, biofuel is flat at 380 and only
same-year comparisons are visible; discounted, a tonne abated late competes with a tonne
abated early.

| EUR(2020)/tCO₂, r = 4.5 % | 2020 | 2030 | 2040 | 2050 |
|---|---|---|---|---|
| biofuel | 380 | 244 | 157 | 101 |
| electrofuel, grid | **undefined** | 1645 | 418 | 203 |
| electrofuel, dedicated wind | 1382 | 505 | 242 | 120 |

The baseline pathway abates *nothing* before 2028 — it emits more than the kerosene it
replaces, so no cost per tonne exists at any price. Dedicated wind fixes that and cuts the
cost by 40 % or more. It never gets under biofuel in the *same* year, which is why biofuel
still ends at its biomass ceiling in both runs. But from 2037 on it is cheaper than biofuel
was in 2025, and that is the trade the optimiser takes: it holds 2030 biofuel at 2 %
against 10 %, emits more early, and buys the abatement back with electrofuel after 2040 —
49 % of the blend by 2050.

In [ ]:
PS.pathway()

### 7.4 Social discount rate

The 3.2 % line is drawn heavier with the baseline dashed over it, because the two designs
are identical to every digit — the vertex result of `01_optimisation_runs.ipynb` §3.

At 15 % panel **b** is the striking one: emissions run *above* every other case from 2026
to 2041 and then collapse to 34 MtCO₂, about a third of the baseline's 97, while panel
**d** shows biofuel flat at 2 % until 2030. It is a deliberate do-nothing-then-panic path.
Panel **c** prices it at +136 EUR/tCO₂ in 2050 against the baseline's +8, and it is the only
run of the fourteen whose 2020–2050 average is positive (+11 against −30) — genuinely more
expensive in real resources — while the *reported* policy cost falls to 18.5 bn € purely
because the discounting erases the decade it lands in. At that rate the objective stops
measuring what the policy costs.

In [ ]:
PS.discount()

### 7.5 Every sensitivity against one baseline

Eight quantities for every run. Besides blocks A–D there are three groups from the
budget × biomass work: five rungs of `main`'s carbon-budget ladder; the three other biomass
allocations and the pessimistic roadmap, each re-optimised at the ReFuelEU-equivalent
budget (`sweep/run_refueleu_budget.py B5 B75 B15 pess`, warm-started from its own 3.2 rung);
and the fossil BAU as the reference. All four new runs converged on KKT, and each sits
between its own 3.2 and 3.0 rungs.

Panel **a** is the objective, discounted to 2020 at each run's own rate, so the rows marked
† are not comparable in level with the rest. The fixed-demand run does not report it; its
value is reconstructed from the model's own formula with the travel term at zero, which is
exact because its traffic is the reference traffic.

What it shows:

* **The carbon budget dominates the objective**: −157 bn€ at 3.8 % to +232 bn€ at 2.2 %,
  against the baseline's −47. At 2.6 % and below the 2050 blend is already at its ceiling
  (biofuel 44 % + electrofuel 55 %) and 2050 traffic is at 2.04 T RPK, so a tighter budget
  can only be met earlier: 2035 electrofuel goes from 4 % at 2.6 to 15 % at 2.2.
* **Biomass sets the fuel mix.** At 5 % the 2050 blend is 22 % biofuel and 54 % electrofuel;
  at 15 % it is 58 % biofuel and no electrofuel at all. The objective moves less, −13 to
  −64 bn€.
* **A pessimistic technology roadmap costs the most of any single change** at the central
  budget: +35 bn€ against −47. It needs electrofuel at 38 % in 2050 and traffic falls to
  2.11 T RPK.
* **Blocks A–D move the mandates far more than the objective.** Across elasticity the
  objective spans −52 to −46 bn€; no ramp-up limit moves it more than 6 bn€ from the
  baseline; dedicated wind lowers
  it to −60 bn€ while taking 2050 electrofuel from 14 % to 49 %.
* **Electrofuel's 2035 mandate is zero in every run except the two tightest budgets**: every
  other case starts it only after 2035.

In [ ]:
PS.summary()

## Not migrated

`supply.pdf` comes from `equilibriums.ipynb`, not from `main.ipynb`. It is the
demand/supply calibration that produced `initial_airfare_per_rpk = 0.09236379319842411`
and depends on `IATA_cost_data.xlsx` rather than on any AeroMAPS model, so the migration
does not touch it and the legacy notebook still runs as written.

`discount_effect.pdf`, `biomass_sensitivity.pdf` and the `overall_colorbar*.pdf` variants
are stale artefacts in the legacy folder: no cell of `main.ipynb` writes them any more.